# O Perceptron — Aula Prática

**Disciplina:** Redes Neurais Artificiais
**Professor:** Robson de Souza Ultramari

Neste notebook vamos implementar do zero o perceptron de Rosenblatt (1958),
treiná-lo nas portas lógicas AND e OR, visualizar a fronteira de decisão e
comprovar experimentalmente a limitação demonstrada por Minsky e Papert (1969)
com a porta XOR.

**Como usar:** execute cada célula com `Shift + Enter`, na ordem.
Nada precisa ser instalado — NumPy e Matplotlib já vêm no Colab.

## Célula 1 — Bibliotecas e conjunto de dados

`numpy` faz as operações com vetores; `matplotlib` desenha os gráficos.

O conjunto de treinamento da porta AND tem 4 amostras, cada uma com 2 atributos.
O vetor `d` guarda a saída desejada de cada amostra.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Conjunto de treinamento da porta AND
X_and = np.array([[0, 0],
                  [0, 1],
                  [1, 0],
                  [1, 1]], dtype=float)

d_and = np.array([0, 0, 0, 1])

print("Entradas (X):")
print(X_and)
print("\nSaidas desejadas (d):", d_and)

## Célula 2 — A classe Perceptron

A classe é a tradução direta das fórmulas vistas na teoria:

| Teoria | Código |
|---|---|
| $u = \mathbf{w}^T\mathbf{x} + b$ | `np.dot(self.w, x) + self.b` |
| $f(u) = 1$ se $u \geq 0$, senão $0$ | `1 if u >= 0 else 0` |
| $e = d - \hat{y}$ | `e = di - y` |
| $w_i \leftarrow w_i + \eta \, e \, x_i$ | `self.w += self.eta * e * xi` |
| $b \leftarrow b + \eta \, e$ | `self.b += self.eta * e` |

Repare que a classe inteira precisa ficar em **uma única célula** — o Python
não permite dividir a definição de uma classe entre células diferentes.

In [ ]:
class Perceptron:
    """Perceptron de camada única com função de ativação degrau."""

    def __init__(self, n_entradas, eta=0.1, epocas=100):
        self.w = np.zeros(n_entradas)   # pesos iniciais
        self.b = 0.0                    # viés inicial
        self.eta = eta                  # taxa de aprendizado
        self.epocas = epocas            # limite de épocas
        self.historico = []             # erros por época

    def ativacao(self, u):
        """Função degrau: converte o valor contínuo u em 0 ou 1."""
        return 1 if u >= 0 else 0

    def prever(self, x):
        """Calcula a soma ponderada e aplica a ativação."""
        u = np.dot(self.w, x) + self.b
        return self.ativacao(u)

    def treinar(self, X, d, verbose=True):
        """Ajusta pesos e viés percorrendo o conjunto até convergir."""
        self.historico = []

        for epoca in range(self.epocas):
            erros = 0

            for xi, di in zip(X, d):
                y = self.prever(xi)     # saída obtida
                e = di - y              # erro: -1, 0 ou +1

                if e != 0:
                    self.w = self.w + self.eta * e * xi
                    self.b = self.b + self.eta * e
                    erros += 1

            self.historico.append(erros)

            if verbose:
                print(f"Época {epoca+1:3d} | erros: {erros} | "
                      f"w = {np.round(self.w, 2)} | b = {self.b:.2f}")

            if erros == 0:
                if verbose:
                    print(f"\nConvergiu na época {epoca+1}.")
                return True

        if verbose:
            print(f"\nNAO convergiu em {self.epocas} épocas.")
        return False

## Célula 3 — Treinando a porta AND

Use `eta=0.5` para reproduzir exatamente a tabela feita à mão no slide.
Confira: os pesos impressos na 1ª e na 2ª época devem bater com os do quadro.

In [ ]:
p_and = Perceptron(n_entradas=2, eta=0.5)
p_and.treinar(X_and, d_and)

print("\nTeste do modelo treinado:")
for xi, di in zip(X_and, d_and):
    print(f"  {xi.astype(int)} -> previsto: {p_and.prever(xi)} | esperado: {di}")

## Célula 4 — Visualizando a fronteira de decisão

A fronteira é a reta $w_1x_1 + w_2x_2 + b = 0$. Isolando $x_2$:

$$x_2 = -\frac{w_1}{w_2}x_1 - \frac{b}{w_2}$$

A função abaixo desenha os pontos do conjunto e essa reta.
É aqui que a turma **vê** o que o treinamento fez: girar e deslocar a reta
até separar as duas classes.

In [ ]:
def plotar_fronteira(modelo, X, d, titulo):
    """Desenha as amostras e a reta de decisão aprendida."""
    plt.figure(figsize=(5, 5))

    # amostras: círculo para classe 0, xis para classe 1
    for xi, di in zip(X, d):
        if di == 1:
            plt.scatter(xi[0], xi[1], marker='x', s=160,
                        c='crimson', linewidths=2.5, label='classe 1')
        else:
            plt.scatter(xi[0], xi[1], marker='o', s=140,
                        facecolors='none', edgecolors='navy',
                        linewidths=2.5, label='classe 0')

    # reta de decisão (só existe se w2 != 0)
    w1, w2 = modelo.w
    b = modelo.b
    if w2 != 0:
        x1 = np.linspace(-0.5, 1.5, 100)
        x2 = -(w1 / w2) * x1 - b / w2
        plt.plot(x1, x2, 'k--', linewidth=2, label='fronteira')
    elif w1 != 0:
        plt.axvline(-b / w1, color='k', linestyle='--',
                    linewidth=2, label='fronteira')

    plt.xlim(-0.5, 1.5)
    plt.ylim(-0.5, 1.5)
    plt.xlabel('x1')
    plt.ylabel('x2')
    plt.title(titulo)
    plt.grid(alpha=0.3)

    # remove entradas repetidas da legenda
    itens = dict(zip(*reversed(plt.gca().get_legend_handles_labels())))
    plt.legend(itens.values(), itens.keys(), loc='upper right')
    plt.show()


plotar_fronteira(p_and, X_and, d_and, 'Porta AND — fronteira aprendida')

## Célula 5 — A porta OR

Mesmo código, apenas trocando o vetor de saídas desejadas.
Compare o número de épocas e os pesos finais com os da porta AND.

In [ ]:
d_or = np.array([0, 1, 1, 1])

p_or = Perceptron(n_entradas=2, eta=0.5)
p_or.treinar(X_and, d_or)

plotar_fronteira(p_or, X_and, d_or, 'Porta OR — fronteira aprendida')

## Célula 6 — A porta XOR não converge

Agora a demonstração central da aula.

A saída desejada do XOR é 1 quando as entradas são **diferentes**.
Não existe uma única reta capaz de separar esses pontos — e o algoritmo,
que só sabe traçar retas, nunca chega a zero erro.

Deixe rodar as 30 épocas e observe os erros oscilando sem parar.

In [ ]:
d_xor = np.array([0, 1, 1, 0])

p_xor = Perceptron(n_entradas=2, eta=0.5, epocas=30)
convergiu = p_xor.treinar(X_and, d_xor)

print("\nConvergiu?", convergiu)

In [ ]:
# Erros por época: AND cai a zero, XOR nunca cai
plt.figure(figsize=(7, 4))
plt.plot(range(1, len(p_and.historico) + 1), p_and.historico,
         'o-', label='AND (converge)')
plt.plot(range(1, len(p_xor.historico) + 1), p_xor.historico,
         's-', label='XOR (não converge)')
plt.xlabel('época')
plt.ylabel('nº de erros')
plt.title('Erros por época')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

plotar_fronteira(p_xor, X_and, d_xor, 'Porta XOR — nenhuma reta separa')

## Exercícios

Resolva criando novas células abaixo.

1. Treine o perceptron para a porta **NAND** (saídas `[1, 1, 1, 0]`).
   Compare os pesos finais com os da porta AND. O que você observa?

2. Altere a taxa de aprendizado para `0.01` e depois para `1.0` na porta AND.
   Quantas épocas foram necessárias em cada caso?

3. Troque a inicialização dos pesos por valores aleatórios
   (`np.random.uniform(-1, 1, 2)`). Rode cinco vezes. Os pesos finais
   são sempre os mesmos? E a fronteira desenhada?

4. Implemente a função sinal (saídas `-1` e `+1`) no lugar da função degrau
   e refaça o treinamento da porta AND com `d = [-1, -1, -1, 1]`.

5. Gere dois grupos de pontos aleatórios linearmente separáveis com
   `np.random.randn` e treine o perceptron sobre eles.

6. **Discussão:** por que a função degrau impede o uso de gradiente
   descendente no treinamento? O que isso tem a ver com a escolha da
   função sigmoide nas redes multicamadas?

## Referências

- ROSENBLATT, F. The perceptron: a probabilistic model for information storage
  and organization in the brain. *Psychological Review*, v. 65, n. 6,
  p. 386–408, 1958. DOI: 10.1037/h0042519.
- McCULLOCH, W. S.; PITTS, W. A logical calculus of the ideas immanent in
  nervous activity. *Bulletin of Mathematical Biophysics*, v. 5, n. 4,
  p. 115–133, 1943.
- NOVIKOFF, A. B. J. On convergence proofs on perceptrons. In: *Proceedings of
  the Symposium on the Mathematical Theory of Automata*, v. XII. Brooklyn:
  Polytechnic Institute of Brooklyn, 1962. p. 615–622.
- MINSKY, M.; PAPERT, S. *Perceptrons: an introduction to computational
  geometry*. Cambridge: MIT Press, 1969.
- HAYKIN, S. *Redes neurais: princípios e prática*. 2. ed. Porto Alegre:
  Bookman, 2001.
- GOODFELLOW, I.; BENGIO, Y.; COURVILLE, A. *Deep Learning*. Cambridge:
  MIT Press, 2016. Disponível em: https://www.deeplearningbook.org.